In [1]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool, ToolRuntime
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

/workspaces/Python/Agent-SSS/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


True

<div style="font: bold 40px sans-serif;">LangChain Middleware 中间件</div>

<br/>

中间件是一种控制Agent内部运行过程的技术，它在智能体运行的各个过程中预留钩子（hook），方便我们嵌入自定义操作。

# Middleware核心功能

- **拦截和修改请求** - 在模型调用前后对输入输出进行处理
- **实现PII脱敏** - 自动检测和脱敏敏感个人信息
- **会话摘要管理** - 当对话过长时自动压缩历史消息
- **人工审核机制** - 在执行危险操作前等待人工确认
- **动态模型选择** - 根据运行时条件选择不同的模型
- **自定义状态管理** - 扩展Agent状态以跟踪额外信息

Agent的默认执行流程和中间件嵌入钩子的对比如图：

<div style="display:flex; justify-content:center; align-items:center; gap:20px;">
<img src="./resources/agents-loop.png" style="max-width:400px; height:auto;" alt="Agents Loop">
<img src="./resources/hooks.png" style="max-width:100%; height:auto;" alt="Hooks">
</div>

如上图所示，LangChain的Middleware基于钩子（Hooks）机制，主要有以下几种类型：

| 钩子类型 | 说明 |
|---------|------|
| `before_model` | 在模型调用之前执行，可修改输入 |
| `after_model` | 在模型调用之后执行，可处理输出 |
| `wrap_model_call` | 包装整个模型调用过程，可完全控制执行流程 |
| `pre_model_hook` | LangChain v0中等效于before_model |
| `post_model_hook` | LangChain v0中等效于after_model |

# 2. 预置中间件（Built-in Middleware）

LangChain提供了多种预置中间件，可以直接使用。

## 2.1 PIIMiddleware - 个人信息脱敏

自动检测并脱敏消息中的个人身份信息（PII），如邮箱、电话号码、身份证号等。
其中，PII的处理策略有四种：
- `'block'` - 抛出异常
- `'redact'` - 用 [REDACTED_{PII_TYPE}] 来替代
- `'mask'` - 关键信息采用**掩码 (例如., `****-****-****-1234`)
- `'hash'` - 用哈希值来替换


In [9]:
from langchain.agents.middleware import PIIMiddleware

# 也可以添加多个PII中间件来检测不同类型的敏感信息
pii_middleware_email = PIIMiddleware(
    "email",
    strategy="redact",  # mask
    apply_to_input=False,
    apply_to_output=True
)

pii_middleware_phone = PIIMiddleware(
    "phone_number",
    # 使用自定义正则表达式
    detector=r"(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{4}",
    strategy="mask",
    apply_to_input=False,
    apply_to_output=True
)

print("PIIMiddleware 配置完成")
print(f"- 邮箱脱敏策略: {pii_middleware_email.strategy}")
print(f"- 电话号码策略: {pii_middleware_phone.strategy}")

PIIMiddleware 配置完成
- 邮箱脱敏策略: redact
- 电话号码策略: mask


In [10]:
import os

chat_model = init_chat_model(
    model="gui-plus",  # 换成你在百炼开通的模型名称，例如 qwen-max
    model_provider="openai",
     api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

agent_with_builtin_middleware = create_agent(
    chat_model,  # 原来的 "deepseek-chat" 改为这个模型对象
    middleware=[pii_middleware_email, pii_middleware_phone]

)

In [11]:
response = agent_with_builtin_middleware.invoke({
    "messages": [
        HumanMessage(
            "我的邮箱是: huge@itcast.cn,我的电话是13698023405,你要记住这些信息，后续要用到。如果记住了请确认一遍。")
    ]
})
for m in response['messages']:
    m.pretty_print()
    
    # 返回的是PIIDetectionError 的话，说明strategy策略设置的block生效了，直接阻断了。                         

================================ Human Message =================================

我的邮箱是: huge@itcast.cn,我的电话是13698023405,你要记住这些信息，后续要用到。如果记住了请确认一遍。
================================== Ai Message ==================================

我已经记住了您的邮箱和电话信息。您的邮箱是huge@itcast.cn，您的电话号码是****3405。


## 2.2 ModelFallbackMiddleware

可以在创建智能体时设置多个模型，如果第一个模型调用失败，会自动调用下一个模型。


In [22]:
from langchain.agents.middleware import ModelFallbackMiddleware

def make_dashscope_model(name):
    return init_chat_model(
        model=name,
        model_provider="openai",
        api_key=os.environ["DASHSCOPE_API_KEY"],
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )


# 本质上还是调用init_chat_model获取langchain规定好的模型。没有的还得和上面一样自己定义
# 这边我们自己做一个函数方便定义模型

model_fallback_middleware = ModelFallbackMiddleware(
    make_dashscope_model("kimi-k3.1-pro"),
    make_dashscope_model("tongyi-intent-detect-v3"),
    make_dashscope_model("qwen3.8-max-0902"),
)
test_primary = make_dashscope_model("fallback-test-model-does-not-exist")

# 这里调用：test_primary，一个不存在的模型，如果还是能成功说明调用了middleware
agent_with_model_fallback = create_agent(
    test_primary,
    middleware=[model_fallback_middleware]
)

In [24]:
response = agent_with_model_fallback.invoke({
    "messages": [HumanMessage("你是谁？")]
})

# 测试检验
last_message = response["messages"][-1]
print(last_message.content)
print(last_message.response_metadata)


for m in response['messages']:
    m.pretty_print()

我是 Kimi，由 Moonshot AI（月之暗面）开发的 AI 助手。有什么可以帮你的吗？
{'token_usage': {'completion_tokens': 144, 'prompt_tokens': 87, 'total_tokens': 231, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 107, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': 87}}, 'model_provider': 'openai', 'model_name': 'kimi-k3.1-pro', 'system_fingerprint': None, 'id': 'chatcmpl-6fb8ffed-e5be-9638-8b6f-b05e40f3bc22', 'finish_reason': 'stop', 'logprobs': None}
================================ Human Message =================================

你是谁？
================================== Ai Message ==================================

我是 Kimi，由 Moonshot AI（月之暗面）开发的 AI 助手。有什么可以帮你的吗？


## 2.3 HumanInTheLoopMiddleware - 人工审核

在执行特定工具调用前暂停，等待人工确认。可用于控制敏感操作如发送邮件、转账等。

In [25]:
emails = [
    {
        "subject": "周末见个面？",
        "content": """
                嗨 虎哥，
                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？

                祝好，简
            """,
        "from": "jane@itcast.cn",
        "status": "unread"
    },
    {
        "subject": "周五会议",
        "content": """
                嗨 虎哥，
                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？

                祝好，小李
            """,
        "from": "lixiaolong@itcast.cn",
        "status": "checked"
    }
]

In [26]:
# 定义工具，收取和发送邮件
@tool
def check_inbox() -> str:
    """查看收件箱中的邮件."""

    # 模拟收件箱邮件
    return str(emails)


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """发送回复邮件"""
    return f"邮件已发送至 {to} , 主题： {subject} , 内容： {body}"


In [27]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

# 创建人工审核中间件
human_in_loop_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "send_email": {  # 要人工确认的工具的名字
            "description": "请确认邮件内容",
            "allowed_decisions": ["approve", "reject"]
        }
    }
)

In [28]:
agent_with_hitl = create_agent(
    chat_model,
    tools=[check_inbox, send_email],
    middleware=[human_in_loop_middleware],
    checkpointer=InMemorySaver(),
    system_prompt="你是一个邮件助手"
)

In [29]:
config = {"configurable": {"thread_id": "1"}}
response = agent_with_hitl.invoke(
    {"messages": [HumanMessage("帮我查看邮件")]},
    config=config
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================
Tool Calls:
  check_inbox (call_2c3618a13cb04bbf8e52d3)
 Call ID: call_2c3618a13cb04bbf8e52d3
  Args:
================================= Tool Message =================================
Name: check_inbox

[{'subject': '周末见个面？', 'content': '\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ', 'from': 'jane@itcast.cn', 'status': 'unread'}, {'subject': '周五会议', 'content': '\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ', 'from': 'lixiaolong@itcast.cn', 'status': 'checked'}]
================================== Ai Message ==================================

这是你的邮件


In [30]:
response = agent_with_hitl.invoke(
    {"messages": [HumanMessage("帮我回复Jane，很高兴她能来，问问她具体见面时间")]},
    config=config
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================
Tool Calls:
  check_inbox (call_2c3618a13cb04bbf8e52d3)
 Call ID: call_2c3618a13cb04bbf8e52d3
  Args:
================================= Tool Message =================================
Name: check_inbox

[{'subject': '周末见个面？', 'content': '\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ', 'from': 'jane@itcast.cn', 'status': 'unread'}, {'subject': '周五会议', 'content': '\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ', 'from': 'lixiaolong@itcast.cn', 'status': 'checked'}]
================================== Ai Message ==================================

这是你的邮件
================================ Human Message =================================

帮我回复Jane，很高兴她能来，问问她具体见面时间
================================== Ai Messa

In [31]:
print(response)

{'messages': [HumanMessage(content='帮我查看邮件', additional_kwargs={}, response_metadata={}, id='c50db99b-3f71-4762-b8b3-0ecf663c2cf7'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 301, 'total_tokens': 315, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': None, 'rejected_prediction_tokens': None, 'text_tokens': 14}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': None, 'image_tokens': None, 'text_tokens': 301}}, 'model_provider': 'openai', 'model_name': 'gui-plus', 'system_fingerprint': None, 'id': 'chatcmpl-55b6ab78-af51-94fb-9f85-a25843e7ccad', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1c956-5efb-7393-baff-5dc32eccf16b-0', tool_calls=[{'name': 'check_inbox', 'args': {}, 'id': 'call_2c3618a13cb04bbf8e52d3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inpu

In [39]:
config_test = {"configurable": {"thread_id": "hitl-test-001"}}

response = agent_with_hitl.invoke(
    {"messages": [HumanMessage(
        content=(
            "请调用 send_email 给 jane@itcast.cn 发邮件，"
            "主题是“周末见面”，正文是“很高兴你能来，具体什么时候见面？”"
        )
    )]},
    config=config_test,
)

interrupts = response.get("__interrupt__", ())
if interrupts:
    print(interrupts[0].value)
else:
    last = response["messages"][-1]
    print("没有触发人工审核")
    print("content:", last.content)
    print("tool_calls:", getattr(last, "tool_calls", None))

{'action_requests': [{'name': 'send_email', 'args': {'to': 'jane@itcast.cn', 'subject': '周末见面', 'body': '很高兴你能来，具体什么时候见面？'}, 'description': '请确认邮件内容'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'reject']}]}


### Reject

In [42]:
from langgraph.types import Command

response = agent_with_hitl.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",  # 人工确认的决定，是reject
                    "message": "语气太书面了，Jane是我的女朋友！"  # 请求被拒绝的原因解释
                }
            ]
        }
    ),
    config=config_test  # 相同的thread_id以恢复暂停的对话
)

In [43]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

请调用 send_email 给 jane@itcast.cn 发邮件，主题是“周末见面”，正文是“很高兴你能来，具体什么时候见面？”
================================== Ai Message ==================================
Tool Calls:
  send_email (call_2393c5f3ce4e4d65988aa5)
 Call ID: call_2393c5f3ce4e4d65988aa5
  Args:
    to: jane@itcast.cn
    subject: 周末见面
    body: 很高兴你能来，具体什么时候见面？
================================= Tool Message =================================
Name: send_email

User rejected the tool call for `send_email` with reason: 语气太书面了，Jane是我的女朋友！
================================== Ai Message ==================================

我理解你的感受，但很抱歉我目前无法直接发送邮件。


### approve



In [44]:
from langgraph.types import Command

response = agent_with_hitl.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "approve"
                }
            ]
        }
    ),
    config=config_test  # 相同的thread_id以恢复暂停的对话
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

请调用 send_email 给 jane@itcast.cn 发邮件，主题是“周末见面”，正文是“很高兴你能来，具体什么时候见面？”
================================== Ai Message ==================================
Tool Calls:
  send_email (call_2393c5f3ce4e4d65988aa5)
 Call ID: call_2393c5f3ce4e4d65988aa5
  Args:
    to: jane@itcast.cn
    subject: 周末见面
    body: 很高兴你能来，具体什么时候见面？
================================= Tool Message =================================
Name: send_email

User rejected the tool call for `send_email` with reason: 语气太书面了，Jane是我的女朋友！
================================== Ai Message ==================================

我理解你的感受，但很抱歉我目前无法直接发送邮件。


# 3. 自定义中间件

除了使用预置中间件，我们还可以利用LangChain提供的hook创建完全自定义的中间件。

根据hook的种类，中间件可以分为两类：
- Node-style hooks：在具体某个节点执行的中间件，包含：
  - before_agent
  - before_model
  - after_model
  - after_agent
- Wrap-style hooks：环绕model或tool调用的中间件，包括：
  - wrap_model_call
  - wrap_tool_call

为了便于开发中间件，LangChain为每一种hook都提供了装饰器，我们只有定义函数并使用装饰器标记即可快速开发中间件。


## 3.1 node-style装饰器

例如，我们要实现模型调用计数功能，可以在每次调用模型后(after_model)记录模型调用次数


In [45]:
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from typing import Any
from langchain.agents.middleware import after_model


# 1.定义自定义state，记录模型调用次数
class CustomAgentState(AgentState):
    """扩展Agent状态，添加自定义字段"""
    model_call_count: int  # 模型调用次数


# 2.定义中间件
@after_model(state_schema=CustomAgentState)
def increment_counter(state: CustomAgentState, runtime: Runtime) -> dict[str, Any]:
    """使用装饰器的after_model钩子 - 增加调用计数"""
    current_count = state.get("model_call_count", 0)
    return {"model_call_count": current_count + 1}

In [46]:
# 创建智能体，设置middleware
agent = create_agent(
    chat_model,
    middleware=[increment_counter],
    checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke(
    {
        "messages": [HumanMessage("你好")]
    },
    config
)

print(response)

{'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='a4f5bf59-dc53-4b7b-83f1-a1e3dd91dfa8'), AIMessage(content='你好！有什么可以帮助你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 20, 'total_tokens': 28, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': None, 'rejected_prediction_tokens': None, 'text_tokens': 8}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': None, 'image_tokens': None, 'text_tokens': 20}}, 'model_provider': 'openai', 'model_name': 'gui-plus', 'system_fingerprint': None, 'id': 'chatcmpl-e21a3427-981f-9ee3-897a-70ebaf49ffa7', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1c97b-fb79-74a1-9328-7edbd60bb451-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 8, 'total_tokens': 28, 'input_token_details': {}, 'output_toke

In [49]:
# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("讲个笑话吧")]
}, config)

print(response)
print(response["model_call_count"])

{'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='a4f5bf59-dc53-4b7b-83f1-a1e3dd91dfa8'), AIMessage(content='你好！有什么可以帮助你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 20, 'total_tokens': 28, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': None, 'rejected_prediction_tokens': None, 'text_tokens': 8}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': None, 'image_tokens': None, 'text_tokens': 20}}, 'model_provider': 'openai', 'model_name': 'gui-plus', 'system_fingerprint': None, 'id': 'chatcmpl-e21a3427-981f-9ee3-897a-70ebaf49ffa7', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1c97b-fb79-74a1-9328-7edbd60bb451-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 8, 'total_tokens': 28, 'input_token_details': {}, 'output_toke

## 3.2.wrap-style装饰器

例如，我们来定义一个可以在调用模型时失败重试的中间件，最大重试次数为3次


In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable


@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            print(f"Retry {attempt + 1}/3 after error: {e}")
            if attempt == 2:
                raise
    return None


agent = create_agent(
    model="deepseek-flash",
    middleware=[retry_model]
)

In [ ]:
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke(
    {"messages": [HumanMessage("Hello")]},
    config,
)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='943db39f-c75d-4233-bf0f-9784fea6d842'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None, 'reasoning_content': 'The user just said "Hello". I should respond warmly and openly, inviting them to share what they\'d like to discuss or ask about.'}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 31, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 28, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '1f7197fd-d9ef-4308-8d3

## 3.3 使用类方式创建中间件

对于需要同时用到多个hooks的更复杂的中间件逻辑，建议使用类方式继承AgentMiddleware。

In [ ]:
from langchain.agents.middleware import AgentMiddleware
from langchain.agents.middleware.types import ModelCallResult, ToolCallRequest
from langgraph.types import Command


class LoggingMiddleware(AgentMiddleware):

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelCallResult:
        try:
            print(f"\n=======About to call model with {len(request.messages)} messages=======")
            return handler(request)
        except Exception as e:
            print(f"\n=======[错误]: {str(e)}=======")
            return AIMessage("调用模型失败，请重试~")

    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command],
    ) -> ToolMessage | Command:
        print(f"\n=======调用工具: {request.tool_call['name']}=======")
        print(f"\n=======参数: {request.tool_call['args']}=======")
        try:
            result = handler(request)
            print("\n=======工具调用成功！=======")
            return result
        except Exception as e:
            print(f"\n=======工具调用失败: {e}=======")
            raise


@tool
def get_weather(location: str):
    """查询指定城市的天气信息"""
    return f"Current weather in {location} is sunny, 25℃."


agent = create_agent(
    model="deepseek-flash",
    middleware=[LoggingMiddleware()],
    tools=[get_weather],
)

In [ ]:
for chunk, metadata in agent.stream(
    {"messages": [HumanMessage("杭州今天天气如何？")]},
    stream_mode="messages"
):
    if chunk and chunk.content:
        print(chunk.content, end="", flush=True)


=======About to call model with 1 messages=======

=======调用工具: get_weather=======

=======参数: {'location': '杭州'}=======

=======工具调用成功！=======
Current weather in 杭州 is sunny, 25℃.
=======About to call model with 3 messages=======
杭州今天天气晴朗，气温约 25℃，体感比较舒适，适合外出活动。☀️

不过早晚温差可能稍大，如果外出时间较晚，建议带一件薄外套。需要我帮你查一下未来几天的天气吗？

# 4. 高级用法

## 4.1 动态修改请求

在wrap_model_call这个hook中，我们可以动态修改任意的request参数，包括：
- model
- tool
- system_prompt
- ...

例如，我们定义中间件，可以根据context中用户的选择来开启或关闭模型的思考模式。


In [ ]:
from langchain.agents.middleware import wrap_model_call
from collections.abc import Callable
from pydantic.dataclasses import dataclass


# 定义一个context，记录用户是否开启思考模式
@dataclass
class UserContext:
    thinking: bool = False


@wrap_model_call
def dynamic_model_settings(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    # 获取上下文参数thinking，是否开启思考模式
    thinking = request.runtime.context.thinking

    # 如果Thinking为True，则开启思考模式，否则关闭思考模式
    thinking_type = "enabled" if thinking else "disabled"
    model_settings = request.model_settings
    model_settings.update()
    model_settings = {"extra_body": {"thinking": {"type": thinking_type}}}

    # 覆盖request中的model参数
    modified_request = request.override(model_settings=model_settings)
    return handler(modified_request)

In [ ]:
# 创建智能体，设置middleware
agent = create_agent(
    model="deepseek-flash",
    middleware=[dynamic_model_settings],
    context_schema=UserContext
)

# 调用智能体，并初始化state
response = agent.invoke(
    {"messages": [HumanMessage("Hello")]},
    context=UserContext(thinking=False)  # 通过Context控制思考模式开关
)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='93503d2d-5409-4dbc-a41c-23068d041c06'), AIMessage(content='Hello! What can I help you with today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 5, 'total_tokens': 15, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'e66bc47c-f8fb-4043-b234-4805f593a926', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ad21-3f69-70a2-8702-dc121c45f71c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 10, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}, 'output_token_d

## 4.2 条件跳转

使用`jump_to`实现在特定条件下跳过模型调用。


例如，我们定义一个统计模型调用次数的中间件，当模型调用次数超过阈值时，就禁止调用，直接结束。

In [ ]:
from langgraph.runtime import Runtime
from typing import Any
from langchain.agents.middleware import AgentMiddleware, hook_config, AgentState, before_model


# 1.定义自定义state，记录模型调用次数
class CustomAgentState(AgentState):
    """扩展Agent状态，添加自定义字段"""
    model_call_count: int  # 模型调用次数


# 2.定义中间件
class ModelCallLimitMiddleware(AgentMiddleware[CustomAgentState]):

    def __init__(self, max_limit: int = 10):
        super().__init__()
        self.max_limit = max_limit

    # after_model hook，模型调用次数计数
    def after_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
        current_count = state.get("model_call_count", 0)
        return {"model_call_count": current_count + 1}

    # before_model hook，校验模型调用次数是否超限
    @hook_config(can_jump_to=["end"])
    def before_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
        # 获取模型调用次数
        current_count = state.get("model_call_count", 0) + 1
        # 判断是否超限
        print(f"当前模型调用次数：{current_count}/{self.max_limit}")
        if current_count > self.max_limit:
            return {
                "jump_to": "end",
                "messages": [AIMessage(f"模型调用次数超过最大限制：{self.max_limit}！")]
            }
        return None

# @before_model(can_jump_to=["end"])
# def check_call_limit(state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
#     pass

In [ ]:
# 创建智能体，设置middleware
agent = create_agent(
    model="deepseek-flash",
    middleware=[ModelCallLimitMiddleware(max_limit=2)],
    checkpointer=InMemorySaver(),
    state_schema=CustomAgentState
)
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke(
    {
        "messages": [HumanMessage("Hello")],
    },
    config
)

print(response)

当前模型调用次数：1/2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='f81495b9-60d2-4427-9659-b9a459e0c7ac'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer to user "Hello". Need likely greet back, maybe ask how can help. Simple.'}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 31, 'total_tokens': 62, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 21, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '974a6f2d-23ed-497e-a7ff-d1e25a9b67eb', 'finish_reas

In [ ]:
# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("tell me a joke")]
}, config)

print(response)

当前模型调用次数：2/2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='f81495b9-60d2-4427-9659-b9a459e0c7ac'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer to user "Hello". Need likely greet back, maybe ask how can help. Simple.'}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 31, 'total_tokens': 62, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 21, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '974a6f2d-23ed-497e-a7ff-d1e25a9b67eb', 'finish_reas

In [ ]:
# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("tell me another joke")]
}, config)

print(response)

当前模型调用次数：3/2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='f81495b9-60d2-4427-9659-b9a459e0c7ac'), AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer to user "Hello". Need likely greet back, maybe ask how can help. Simple.'}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 31, 'total_tokens': 62, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 21, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '974a6f2d-23ed-497e-a7ff-d1e25a9b67eb', 'finish_reas

---

**更多资源**:
- [LangChain官方文档](https://docs.langchain.com/oss/python/langchain/middleware/custom)
- [LangChain GitHub](https://github.com/langchain-ai/langchain)